# Tier 4 Solutions — Set 3: Graphs and Dynamic Programming

**Modules covered**: Graph Traversal, Shortest Paths, MST, Topological Sort, Memoization, Tabulation, Knapsack, Sequence Alignment

---

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook builds algorithmic thinking used to reason about performance and correctness in computational biology tools.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Big-O is asymptotic guidance; constant factors still matter in practice.
- Correctness comes before optimization: verify edge cases before performance tuning.
- Data structure choice often dominates speed more than micro-optimizations.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


## Problem 1: Graph from Edge List (1 star)

In [ ]:
from collections import defaultdict


def build_graph(edges: list[tuple[str, str]]) -> dict[str, list[str]]:
    graph: dict[str, list[str]] = defaultdict(list)
    for u, v in edges:
        graph[u].append(v)
        graph[v].append(u)
    return dict(graph)


ppi_edges = [
    ("TP53", "MDM2"),
    ("TP53", "BRCA1"),
    ("TP53", "ATM"),
    ("TP53", "CDKN1A"),
    ("MDM2", "BRCA1"),
    ("MDM2", "RB1"),
    ("BRCA1", "BARD1"),
    ("ATM", "CHEK2"),
    ("CHEK2", "CDKN1A"),
]

graph = build_graph(ppi_edges)

print(f"Vertices: {len(graph)}")
print(f"Edges: {len(ppi_edges)}")
print("Degree distribution (descending):")
for v, neighbors in sorted(graph.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"  {v:<10}: {len(neighbors)}")

## Problem 2: BFS Shortest Path (1 star)

In [ ]:
from collections import deque


def bfs_shortest_path(
    graph: dict[str, list[str]], start: str, end: str
) -> list[str] | None:
    if start not in graph or end not in graph:
        return None
    if start == end:
        return [start]
    visited = {start}
    # Queue entries: current node, path so far
    queue = deque([(start, [start])])
    while queue:
        node, path = queue.popleft()
        for neighbor in graph.get(node, []):
            if neighbor == end:
                return path + [neighbor]
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))
    return None


path = bfs_shortest_path(graph, "BARD1", "CDKN1A")
if path:
    print(f"Shortest path from BARD1 to CDKN1A:")
    print(f"  {' -> '.join(path)}  (length {len(path) - 1})")
else:
    print("No path found.")

path2 = bfs_shortest_path(graph, "BARD1", "UNKNOWNGENE")
print(f"\nPath to UNKNOWNGENE: {path2}")

## Problem 3: Connected Components (2 stars)

In [ ]:
def connected_components(graph: dict[str, list[str]]) -> list[list[str]]:
    visited = set()
    components = []

    def dfs(node: str, component: list) -> None:
        visited.add(node)
        component.append(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                dfs(neighbor, component)

    for node in graph:
        if node not in visited:
            component: list[str] = []
            dfs(node, component)
            components.append(sorted(component))

    return sorted(components, key=len, reverse=True)


extended_edges = ppi_edges + [
    ("KRAS", "BRAF"),
    ("BRAF", "MEK1"),
]

extended_graph = build_graph(extended_edges)
components = connected_components(extended_graph)

print(f"Connected components: {len(components)}")
print(f"Sizes: {[len(c) for c in components]}")
print(f"Largest component: {components[0]}")

## Problem 4: Fibonacci with Memoization (2 stars)

In [ ]:
import time


def fib_naive(n: int) -> int:
    if n <= 1:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)


def fib_memo(n: int, memo: dict | None = None) -> int:
    if memo is None:
        memo = {}
    if n in memo:
        return memo[n]
    if n <= 1:
        return n
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]


def fib_tab(n: int) -> int:
    if n <= 1:
        return n
    dp = [0] * (n + 1)
    dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n]


n = 30
for name, func in [("naive", fib_naive), ("memoized", fib_memo), ("tabulation", fib_tab)]:
    t0 = time.perf_counter()
    result = func(n)
    elapsed = time.perf_counter() - t0
    print(f"{name:<12}: fib({n}) = {result}  time={elapsed:.4f}s")

## Problem 5: Edit Distance with Operations (2 stars)

In [ ]:
def edit_distance_with_ops(source: str, target: str) -> tuple[int, list[str]]:
    m, n = len(source), len(target)
    # dp[i][j] = edit distance between source[:i] and target[:j]
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if source[i - 1] == target[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j - 1],  # substitute
                                    dp[i - 1][j],       # delete
                                    dp[i][j - 1])       # insert

    # Traceback
    ops = []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0 and source[i - 1] == target[j - 1]:
            ops.append(f"Match     {source[i-1]} -> {target[j-1]}")
            i -= 1; j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
            ops.append(f"Substitute {source[i-1]} -> {target[j-1]}")
            i -= 1; j -= 1
        elif i > 0 and dp[i][j] == dp[i - 1][j] + 1:
            ops.append(f"Delete    {source[i-1]}")
            i -= 1
        else:
            ops.append(f"Insert       {target[j-1]}")
            j -= 1

    return dp[m][n], list(reversed(ops))


pairs = [
    ("ATCGTA", "ACGTTA"),
    ("KITTEN", "SITTING"),
    ("GATTACA", "GCATGCU"),
]

for source, target in pairs:
    dist, ops = edit_distance_with_ops(source, target)
    print(f"Source: {source}")
    print(f"Target: {target}")
    print(f"Edit distance: {dist}")
    print("Operations:")
    for op in ops:
        print(f"  {op}")
    print()

## Problem 6: Dijkstra on Metabolic Network (3 stars)

In [ ]:
import heapq


def dijkstra(
    graph: dict[str, list[tuple[str, float]]], start: str, end: str
) -> tuple[float, list[str]] | tuple[None, None]:
    # dist[node] = best known cost to reach node
    dist: dict[str, float] = {start: 0.0}
    prev: dict[str, str | None] = {start: None}
    heap = [(0.0, start)]

    while heap:
        cost, node = heapq.heappop(heap)
        if cost > dist.get(node, float("inf")):
            continue  # stale entry
        if node == end:
            # Reconstruct path
            path = []
            cur: str | None = end
            while cur is not None:
                path.append(cur)
                cur = prev.get(cur)
            return cost, list(reversed(path))
        for neighbor, weight in graph.get(node, []):
            new_cost = cost + weight
            if new_cost < dist.get(neighbor, float("inf")):
                dist[neighbor] = new_cost
                prev[neighbor] = node
                heapq.heappush(heap, (new_cost, neighbor))

    return None, None


metabolic_network = {
    "Glucose":  [("G6P", 0.5)],
    "G6P":      [("F6P", 0.5), ("G1P", 1.2)],
    "F6P":      [("F1,6BP", 1.0)],
    "F1,6BP":   [("DHAP", 0.3), ("G3P", 0.3)],
    "DHAP":     [("G3P", 0.2)],
    "G3P":      [("3PG", 1.5)],
    "3PG":      [("2PG", 0.4)],
    "2PG":      [("PEP", 0.6)],
    "PEP":      [("Pyruvate", 1.0)],
    "G1P":      [("UDP-Glucose", 2.0)],
    "UDP-Glucose": [],
    "Pyruvate": [],
}

cost, path = dijkstra(metabolic_network, "Glucose", "Pyruvate")
if path:
    print(f"Shortest pathway from Glucose to Pyruvate:")
    print(f"  {' -> '.join(path)}")
    print(f"  Total flux cost: {cost:.1f}")
else:
    print("No pathway found.")

## Problem 7: LCS of Multiple Sequences (3 stars)

In [ ]:
def lcs_two(s1: str, s2: str) -> str:
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    # Traceback
    result = []
    i, j = m, n
    while i > 0 and j > 0:
        if s1[i - 1] == s2[j - 1]:
            result.append(s1[i - 1])
            i -= 1; j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return "".join(reversed(result))


def lcs_three(s1: str, s2: str, s3: str) -> str:
    l, m, n = len(s1), len(s2), len(s3)
    # 3D DP table
    dp = [[[0] * (n + 1) for _ in range(m + 1)] for _ in range(l + 1)]
    for i in range(1, l + 1):
        for j in range(1, m + 1):
            for k in range(1, n + 1):
                if s1[i - 1] == s2[j - 1] == s3[k - 1]:
                    dp[i][j][k] = dp[i - 1][j - 1][k - 1] + 1
                else:
                    dp[i][j][k] = max(dp[i - 1][j][k], dp[i][j - 1][k], dp[i][j][k - 1])
    # Traceback
    result = []
    i, j, k = l, m, n
    while i > 0 and j > 0 and k > 0:
        if s1[i - 1] == s2[j - 1] == s3[k - 1]:
            result.append(s1[i - 1])
            i -= 1; j -= 1; k -= 1
        elif dp[i - 1][j][k] >= dp[i][j - 1][k] and dp[i - 1][j][k] >= dp[i][j][k - 1]:
            i -= 1
        elif dp[i][j - 1][k] >= dp[i][j][k - 1]:
            j -= 1
        else:
            k -= 1
    return "".join(reversed(result))


seq1 = "ATCGTAGC"
seq2 = "ATGCATGC"
seq3 = "ACGTACGT"

lcs2 = lcs_two(seq1, seq2)
lcs3 = lcs_three(seq1, seq2, seq3)

print(f"seq1: {seq1}")
print(f"seq2: {seq2}")
print(f"seq3: {seq3}")
print(f"LCS (2-seq, s1+s2): {lcs2!r}  (length {len(lcs2)})")
print(f"LCS (3-seq):        {lcs3!r}  (length {len(lcs3)})")

## Problem 8: Optimal BST (3 stars)

In [ ]:
def optimal_bst(keys: list[str], freq: list[float]) -> tuple[float, str]:
    n = len(keys)
    # cost[i][j] = minimum expected search cost for keys[i..j]
    # weight[i][j] = sum of frequencies for keys[i..j]
    # root[i][j] = index of the optimal root for keys[i..j]
    cost = [[0.0] * n for _ in range(n)]
    weight = [[0.0] * n for _ in range(n)]
    root = [[0] * n for _ in range(n)]

    # Base case: single keys
    for i in range(n):
        cost[i][i] = freq[i]
        weight[i][i] = freq[i]
        root[i][i] = i

    # Fill for increasing chain lengths
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            weight[i][j] = weight[i][j - 1] + freq[j]
            cost[i][j] = float("inf")
            for r in range(i, j + 1):
                left_cost  = cost[i][r - 1] if r > i else 0.0
                right_cost = cost[r + 1][j] if r < j else 0.0
                c = left_cost + right_cost + weight[i][j]
                if c < cost[i][j]:
                    cost[i][j] = c
                    root[i][j] = r

    return cost[0][n - 1], keys[root[0][n - 1]]


keys  = ["ATM", "BRCA1", "KRAS", "MYC", "TP53"]
freqs = [0.05,   0.25,    0.10,   0.20,  0.40]

cost, root = optimal_bst(keys, freqs)
print(f"Keys:        {keys}")
print(f"Frequencies: {freqs}")
print(f"Optimal root: {root}")
print(f"Optimal BST expected cost: {cost:.2f}")